In [1]:
import torch
from torch.utils.data import DataLoader
from model import ReportGenerator
from train import ChestXrayDataset, collate_fn
from transformers import AutoTokenizer
from tqdm import tqdm
import json
from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer
import numpy as np
import matplotlib.pyplot as plt
import os
from torchvision import transforms
from fpdf import FPDF  # Import for PDF generation


c:\Users\aceadmin\Desktop\CSCN8010\venv\pytorch_cpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

def plot_and_save(data, title, xlabel, ylabel, filename):
    plt.figure()
    plt.hist(data, bins=30, alpha=0.7)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(True)
    plt.tight_layout()
    os.makedirs('ChestXray/plots', exist_ok=True)
    plt.savefig(f'ChestXray/plots/{filename}')
    plt.close()



In [3]:
def generate_pdf_report(findings, impression, filename="report.pdf"):
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()

    pdf.set_font('Arial', 'B', 16)
    pdf.cell(200, 10, txt="Radiology Report", ln=True, align='C')

    pdf.ln(10)  # Line break
    pdf.set_font('Arial', '', 12)
    pdf.multi_cell(0, 10, f"Findings:\n{findings}")
    pdf.ln(5)
    pdf.multi_cell(0, 10, f"Impression:\n{impression}")

    pdf.output(filename)




In [4]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained('t5-small')

# Add required transform to convert PIL image to tensor
transform = transforms.Compose([transforms.ToTensor()])


In [6]:
from PIL import Image

In [8]:

# Assuming `test_image` is the path to the test image
test_image_path = r".\images\images_normalized\2_IM-0652-1001.dcm.png"  # Replace with actual path
img = Image.open(test_image_path)
img = transform(img).unsqueeze(0).to(device)  # Add batch dimension

# Dummy ground truth (for testing purposes, you can adjust this)
true_f = "Ground truth findings text."
true_i = "Ground truth impression text."



In [11]:
# Create a dataset with only the image for inference
ds = ChestXrayDataset(
    './processed/test.jsonl',
    tokenizer,
    label_map_mesh=None,
    label_map_problems=None,
    transform=transform
)


In [12]:
dl = DataLoader(ds, batch_size=1, shuffle=False, collate_fn=collate_fn)

model = ReportGenerator(embed_dim=512, decoder_name='t5-small', num_labels=1).to(device)

In [14]:
# Load pretrained weights
state_dict = torch.load('./best_model.pt', map_location=device)
state_dict = {k: v for k, v in state_dict.items() if not k.startswith('classifier.')}
model.load_state_dict(state_dict, strict=False)
model.eval()

ReportGenerator(
  (img_encoder): DualImageEncoder(
    (encoder_frontal): ResNet(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (1): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=Fals

In [15]:

bleu_scores_f, bleu_scores_i = [], []
rouge_scores_f, rouge_scores_i = [], []
sample_outputs = []

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

start_token_id = tokenizer.pad_token_id  # Usually 0 for T5

In [16]:

with torch.no_grad():
    img_f = img  # This is the image we want to run inference on
    img_l = img  # Use the same image for the label (if applicable)

    decoder_input_ids = torch.tensor([[start_token_id]]).to(device)

    # Generate findings
    outputs_f, _ = model(img_f, img_l, decoder_input_ids=decoder_input_ids)
    pred_f_ids = torch.argmax(outputs_f.logits, dim=-1)
    pred_f = tokenizer.decode(pred_f_ids[0], skip_special_tokens=True)

    # Generate impression
    outputs_i, _ = model(img_f, img_l, decoder_input_ids=decoder_input_ids)
    pred_i_ids = torch.argmax(outputs_i.logits, dim=-1)
    pred_i = tokenizer.decode(pred_i_ids[0], skip_special_tokens=True)

    # Ground truth (if available)
    true_f = true_f
    true_i = true_i

    # BLEU
    bleu_scores_f.append(sentence_bleu([true_f.split()], pred_f.split()))
    bleu_scores_i.append(sentence_bleu([true_i.split()], pred_i.split()))

    # ROUGE-L
    rouge_scores_f.append(scorer.score(true_f, pred_f)['rougeL'].fmeasure)
    rouge_scores_i.append(scorer.score(true_i, pred_i)['rougeL'].fmeasure)

    # Save sample output (can be printed later)
    sample_outputs.append({
        'Findings True': true_f,
        'Findings Pred': pred_f,
        'Impression True': true_i,
        'Impression Pred': pred_i
    })

RuntimeError: Given groups=1, weight of size [64, 3, 7, 7], expected input[1, 1, 2048, 2048] to have 3 channels, but got 1 channels instead

In [ ]:
# Report metrics
print(f'Findings BLEU: {np.mean(bleu_scores_f):.4f}, ROUGE-L: {np.mean(rouge_scores_f):.4f}')
print(f'Impression BLEU: {np.mean(bleu_scores_i):.4f}, ROUGE-L: {np.mean(rouge_scores_i):.4f}')


In [ ]:
# Print sample predictions
for ex in sample_outputs:
    print('---')
    print('Findings True:', ex['Findings True'])
    print('Findings Pred:', ex['Findings Pred'])
    print('Impression True:', ex['Impression True'])
    print('Impression Pred:', ex['Impression Pred'])



In [ ]:
# Save results to a PDF
generate_pdf_report(pred_f, pred_i, filename="radiology_report.pdf")